In [ ]:
!pip install face_recognition ipywidgets

In [ ]:
import base64
import sqlite3
import datetime
import numpy as np
import cv2
import face_recognition
import pandas as pd
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
import ipywidgets as widgets
from io import BytesIO
from PIL import Image

# --- 1. Database Setup ---
DB_NAME = "attendance_system.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS users
                 (name TEXT PRIMARY KEY, encoding BLOB)''')
    c.execute('''CREATE TABLE IF NOT EXISTS attendance
                 (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT, time TIMESTAMP)''')
    conn.commit()
    conn.close()

init_db()

# --- 2. Camera Helper (JavaScript) ---
def take_photo(quality=0.8):
    js = Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = 'Capture';
        div.appendChild(capture);

        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});

        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // Resize the output to fit the video element.
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        // Wait for Capture to be clicked.
        await new Promise((resolve) => capture.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = base64.b64decode(data.split(',')[1])
    img_np = np.frombuffer(binary, dtype=np.uint8)
    img = cv2.imdecode(img_np, cv2.IMREAD_COLOR)
    return img

# --- 3. Core Logic Functions ---

def register_face(name):
    if not name:
        return "❌ Error: Please enter a name."

    print(f"📸 Capturing face for {name}...")
    try:
        img = take_photo()
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        encodings = face_recognition.face_encodings(rgb_img)

        if not encodings:
            return "⚠️ No face detected. Try again."

        encoding = encodings[0].tobytes()

        conn = sqlite3.connect(DB_NAME)
        c = conn.cursor()
        try:
            c.execute("INSERT INTO users (name, encoding) VALUES (?, ?)", (name, encoding))
            conn.commit()
            msg = f"✅ Success: {name} registered!"
        except sqlite3.IntegrityError:
            msg = f"⚠️ User {name} already exists."
        finally:
            conn.close()
        return msg
    except Exception as e:
        return f"❌ Error: {e}"

def mark_attendance():
    print("📸 Looking for faces...")
    try:
        img = take_photo()
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Load known faces
        conn = sqlite3.connect(DB_NAME)
        c = conn.cursor()
        c.execute("SELECT name, encoding FROM users")
        rows = c.fetchall()

        known_encodings = [np.frombuffer(row[1], dtype=np.float64) for row in rows]
        known_names = [row[0] for row in rows]

        # Detect current faces
        face_locations = face_recognition.face_locations(rgb_img)
        face_encodings = face_recognition.face_encodings(rgb_img, face_locations)

        if not face_encodings:
            return "⚠️ No face detected."

        logs = []
        for face_encoding in face_encodings:
            matches = face_recognition.compare_faces(known_encodings, face_encoding)
            name = "Unknown"

            if True in matches:
                first_match_index = matches.index(True)
                name = known_names[first_match_index]

                # Check duplicate entry for today
                today_str = datetime.datetime.now().strftime("%Y-%m-%d")
                c.execute("SELECT * FROM attendance WHERE name=? AND date(time)=?", (name, today_str))
                if not c.fetchone():
                    c.execute("INSERT INTO attendance (name, time) VALUES (?, ?)", (name, datetime.datetime.now()))
                    conn.commit()
                    logs.append(f"✅ Marked: {name}")
                else:
                    logs.append(f"ℹ️ Already Marked: {name}")
            else:
                logs.append("❓ Unknown Person")

        conn.close()
        return "\n".join(logs)

    except Exception as e:
        return f"❌ Error: {e}"

def get_logs():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query("SELECT * FROM attendance ORDER BY time DESC", conn)
    conn.close()
    return df

In [ ]:
# --- UI Layout ---

# Header
header = widgets.HTML("<h2>📷 Face Recognition Attendance System</h2>")

# 1. Registration Section
reg_label = widgets.Label("New User Registration:")
name_input = widgets.Text(placeholder="Enter Name Here")
btn_register = widgets.Button(description="Capture & Register", button_style='primary', icon='camera')

# 2. Attendance Section
att_label = widgets.Label("Attendance:")
btn_mark = widgets.Button(description="Mark Attendance", button_style='success', icon='check')

# 3. Logs Section
log_label = widgets.Label("Database Logs:")
btn_view_logs = widgets.Button(description="Refresh Logs", button_style='info', icon='table')

# Output Areas (To keep the UI clean)
out_console = widgets.Output(layout={'border': '1px solid #ddd', 'height': '200px', 'overflow_y': 'scroll'})
out_logs = widgets.Output()

# --- Event Handlers ---

def on_register_click(b):
    with out_console:
        clear_output()
        name = name_input.value
        if name:
            msg = register_face(name)
            print(msg)
        else:
            print("⚠️ Please enter a name first.")

def on_mark_click(b):
    with out_console:
        clear_output()
        msg = mark_attendance()
        print(msg)

def on_logs_click(b):
    with out_logs:
        clear_output()
        df = get_logs()
        if not df.empty:
            display(df)
        else:
            print("No attendance records found.")

# Connect buttons to functions
btn_register.on_click(on_register_click)
btn_mark.on_click(on_mark_click)
btn_view_logs.on_click(on_logs_click)

# --- Assemble the Dashboard ---
ui = widgets.VBox([
    header,
    widgets.HTML("<hr>"),

    # Registration Row
    widgets.HBox([reg_label, name_input, btn_register]),
    widgets.HTML("<br>"),

    # Attendance Row
    widgets.HBox([att_label, btn_mark]),
    widgets.HTML("<hr>"),

    # Status/Camera Output
    widgets.Label("System Status / Camera Feed:"),
    out_console,
    widgets.HTML("<hr>"),

    # Logs Row
    widgets.HBox([log_label, btn_view_logs]),
    out_logs
])

# Launch
display(ui)